# 05 - Feature Engineering and Aggregation

In [7]:
# Import LIbraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# Display all columns
pd.set_option('display.max_columns', None)

In [8]:
# Read merged data from Parquet file
fmv_df = pd.read_parquet("fmv_df.parquet")

fmv_df.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimAmount
0,1.0,1,0.10,D,5,0,55,50,B12,'Regular',1217,R82,NaN
1,3.0,1,0.77,D,5,0,55,50,B12,'Regular',1217,R82,NaN
2,5.0,1,0.75,B,6,2,52,50,B12,'Diesel',54,R22,NaN
3,10.0,1,0.09,B,7,0,46,50,B12,'Diesel',76,R72,NaN
4,11.0,1,0.84,B,7,0,46,50,B12,'Diesel',76,R72,NaN


In [9]:
# Split the data into training and testing data
train_df, test_df = train_test_split(fmv_df, test_size=0.3, random_state=42)

# Exposure is not a risk factor. Rather, it is a scaling factor, so it is added as an offset
# What is being modeled is claims by unit exposure
# Exposure is also logged, since the model is only linear on a log scale 
train_df["logExposure"] = np.log(train_df["Exposure"])


In [10]:
cols = ["ClaimNb", "DrivAge", "VehAge", "BonusMalus", "Density", "VehPower", "Area", "Exposure", "ClaimAmount"]
df_model = train_df[cols].copy()

# Keep valid rows
df_model = df_model[(df_model["Exposure"] > 0) & (df_model["Density"] > 0)].copy()

# downcast numerics to reduce memory pressure
df_model["ClaimNb"] = pd.to_numeric(df_model["ClaimNb"], downcast="integer")
df_model["DrivAge"] = pd.to_numeric(df_model["DrivAge"], downcast="integer")
df_model["VehAge"] = pd.to_numeric(df_model["VehAge"], downcast="integer")
df_model["BonusMalus"] = pd.to_numeric(df_model["BonusMalus"], downcast="integer")
df_model["Density"] = pd.to_numeric(df_model["Density"], downcast="float")
df_model["Exposure"] = pd.to_numeric(df_model["Exposure"], downcast="float")
df_model["ClaimAmount"] = pd.to_numeric(df_model["ClaimAmount"], downcast="float")

# convert true categorical fields to category dtype
df_model["VehPower"] = df_model["VehPower"].astype("category")
df_model["Area"] = df_model["Area"].astype("category")

# bin continuous variables
df_model["DrivAgeBand"] = pd.cut(df_model["DrivAge"], bins=[17, 25, 35, 45, 55, 65, 120])
df_model["VehAgeBand"] = pd.cut(df_model["VehAge"], bins=[-1, 1, 3, 5, 10, 20, 100])
df_model["BonusMalusBand"] = pd.cut(df_model["BonusMalus"], bins=[0, 50, 60, 70, 80, 100, 200])
df_model["DensityBand"], density_bins = pd.qcut(df_model["Density"], q=10, duplicates="drop", retbins=True)


df_agg = (
    df_model
    .groupby(
        ["DrivAgeBand", "VehAgeBand", "BonusMalusBand", "DensityBand", "VehPower", "Area"],
        as_index=False,
        observed=True,
        sort=False
    )
    .agg(
        ClaimNb=("ClaimNb", "sum"),
        Exposure=("Exposure", "sum"),
        ClaimAmount=("ClaimAmount", "sum")
    )
)

print("Original rows:", len(df_model))
print("Aggregated rows:", len(df_agg))
print(df_agg.head())


Original rows: 475659
Aggregated rows: 20109
  DrivAgeBand VehAgeBand BonusMalusBand       DensityBand VehPower Area  \
0   (65, 120]   (10, 20]        (0, 50]      (33.0, 67.0]        4    A   
1    (25, 35]    (5, 10]        (0, 50]    (214.0, 393.0]        5    C   
2    (35, 45]    (-1, 1]        (0, 50]  (1313.0, 2715.0]        7    E   
3    (25, 35]     (1, 3]       (70, 80]  (2715.0, 4128.0]        5    E   
4    (55, 65]     (3, 5]        (0, 50]     (0.999, 33.0]        6    A   

   ClaimNb    Exposure  ClaimAmount  
0        6  140.919998     29982.81  
1        2   67.500000      1782.43  
2        6   29.200001      1623.56  
3        4   30.270000     33498.01  
4        9  120.629997      1967.93  


In [11]:
# Store files in Parquet format for use in later scripts

# Training Data
train_df.to_parquet("train_df.parquet")

# Testing Data
test_df.to_parquet("test_df.parquet")

# Save df_model to a Pickle file
df_model.to_pickle("df_model.pickle")


In [12]:
# FREQUENCY GLM

# Fit GLM to aggregated data - Frequency

model_freq = smf.glm(
    formula="ClaimNb ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) + C(DensityBand) + C(VehPower) + C(Area)",
    data=df_agg,
    family=sm.families.NegativeBinomial(alpha=1.0),
    exposure=df_agg["Exposure"]
).fit()

print(model_freq.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:                20109
Model:                            GLM   Df Residuals:                    20068
Model Family:        NegativeBinomial   Df Model:                           40
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -22986.
Date:                Thu, 15 Oct 2026   Deviance:                       11817.
Time:                        00:10:21   Pearson chi2:                 2.84e+04
No. Iterations:                     8   Pseudo R-squ. (CS):            0.09906
Covariance Type:            nonrobust                                         
                                                                  coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------

In [13]:
# Some levels of Density, Vehicle Age, and Area are not significantly different from Reference
# Some of these could be combined with other levels

# Start with Density
density_summary = (
    df_agg.groupby("DensityBand", observed=True)
    .agg(
        claims=("ClaimNb", "sum"),
        exposure=("Exposure", "sum")
    )
    .reset_index()
)

density_summary["frequency"] = density_summary["claims"] / density_summary["exposure"]

print(density_summary.sort_values("DensityBand"))

# Get Coefficients
coef_table = model_freq.summary2().tables[1].reset_index()
coef_table.columns = ["term", "coef", "std_err", "z", "p_value", "ci_lower", "ci_upper"]

density_coef = coef_table[coef_table["term"].str.contains("DensityBand", na=False)]
print(density_coef)

         DensityBand  claims      exposure  frequency
0      (0.999, 33.0]    2394  29219.523438   0.081932
1       (33.0, 67.0]    2574  27927.494141   0.092167
2      (67.0, 122.0]    2503  26153.281250   0.095705
3     (122.0, 214.0]    2623  25701.939453   0.102055
4     (214.0, 393.0]    2615  25607.349609   0.102119
5     (393.0, 720.0]    2799  25077.855469   0.111612
6    (720.0, 1313.0]    3444  25558.593750   0.134749
7   (1313.0, 2715.0]    2910  23138.119141   0.125766
8   (2715.0, 4128.0]    3015  22533.527344   0.133801
9  (4128.0, 27000.0]    3054  20768.363281   0.147051
                                                 term      coef   std_err  \
16  C(DensityBand)[T.Interval(33.0, 67.0, closed='...  0.022538  0.065513   
17  C(DensityBand)[T.Interval(67.0, 122.0, closed=...  0.061215  0.094295   
18  C(DensityBand)[T.Interval(122.0, 214.0, closed...  0.113051  0.116471   
19  C(DensityBand)[T.Interval(214.0, 393.0, closed...  0.064583  0.116172   
20  C(DensityBand)[T.

In [14]:
# Combine Density where frequency and p-values are similar enough

df_agg = df_agg.copy()

bands = list(df_agg["DensityBand"].cat.categories)

df_agg["DensityBand_combined"] = df_agg["DensityBand"].astype(str)

# Bands to group
group_1 = [str(b) for b in bands[0:3]]   # first 3 bands
group_2 = [str(b) for b in bands[3:7]]   # next 4 bands
group_3 = [str(b) for b in bands[7:10]]  # last 3 bands

print("group_1:", group_1)
print("group_2:", group_2)
print("group_3:", group_3)

df_agg.loc[df_agg["DensityBand_combined"].isin(group_1), "DensityBand_combined"] = "Group_1"
df_agg.loc[df_agg["DensityBand_combined"].isin(group_2), "DensityBand_combined"] = "Group_2"
df_agg.loc[df_agg["DensityBand_combined"].isin(group_3), "DensityBand_combined"] = "Group_3"

print(
    df_agg[["DensityBand", "DensityBand_combined"]]
    .drop_duplicates()
    .sort_values("DensityBand_combined")
)

group_1: ['(0.999, 33.0]', '(33.0, 67.0]', '(67.0, 122.0]']
group_2: ['(122.0, 214.0]', '(214.0, 393.0]', '(393.0, 720.0]', '(720.0, 1313.0]']
group_3: ['(1313.0, 2715.0]', '(2715.0, 4128.0]', '(4128.0, 27000.0]']
          DensityBand DensityBand_combined
0        (33.0, 67.0]              Group_1
4       (0.999, 33.0]              Group_1
13      (67.0, 122.0]              Group_1
1      (214.0, 393.0]              Group_2
5     (720.0, 1313.0]              Group_2
7      (393.0, 720.0]              Group_2
25     (122.0, 214.0]              Group_2
2    (1313.0, 2715.0]              Group_3
3    (2715.0, 4128.0]              Group_3
28  (4128.0, 27000.0]              Group_3


In [15]:
# Refit the model with Density combined
model_freq = smf.glm(
    formula="ClaimNb ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) + C(DensityBand_combined) + C(VehPower) + C(Area)",
    data=df_agg,
    family=sm.families.NegativeBinomial(alpha=1.0),
    exposure=df_agg["Exposure"]
).fit()

print(model_freq.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:                20109
Model:                            GLM   Df Residuals:                    20075
Model Family:        NegativeBinomial   Df Model:                           33
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -22989.
Date:                Thu, 15 Oct 2026   Deviance:                       11822.
Time:                        00:10:22   Pearson chi2:                 2.83e+04
No. Iterations:                     8   Pseudo R-squ. (CS):            0.09884
Covariance Type:            nonrobust                                         
                                                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------

In [16]:
# Density still has high p-values. Let's consider removing them. 
# But let's first check the potential impact of removing density 

# With Density
model_freq_with_density = smf.glm(
    formula="ClaimNb ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) + C(DensityBand_combined) + C(VehPower) + C(Area)",
    data=df_agg,
    family=sm.families.NegativeBinomial(alpha=1.0),
    exposure=df_agg["Exposure"]
).fit()

# Without Density
model_freq_ex_density = smf.glm(
    formula="ClaimNb ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) + C(VehPower) + C(Area)",
    data=df_agg,
    family=sm.families.NegativeBinomial(alpha=1.0),
    exposure=df_agg["Exposure"]
).fit()

print(model_freq_ex_density.summary())

# Likelihood Ratio Test and P-value could be informative
lr_stat = 2 * (model_freq_with_density.llf - model_freq_ex_density.llf)
df_diff = model_freq_with_density.df_model - model_freq_ex_density.df_model
p_value = chi2.sf(lr_stat, df_diff)

print("LR stat:", lr_stat)
print("df diff:", df_diff)
print("p-value:", p_value)

# The resulting p-value is high, suggesting density is not statistically significant 
# and would not contribute to the predictions. The LR Test also produced a very small value. 
# So, density will be dropped.

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:                20109
Model:                            GLM   Df Residuals:                    20077
Model Family:        NegativeBinomial   Df Model:                           31
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -22989.
Date:                Thu, 15 Oct 2026   Deviance:                       11822.
Time:                        00:10:24   Pearson chi2:                 2.83e+04
No. Iterations:                     8   Pseudo R-squ. (CS):            0.09880
Covariance Type:            nonrobust                                         
                                                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------

In [17]:
# Now, we examine Vehicle Power

# Vehicle Power 8, 12, 13, 14, 15 seem to be insignificant based on their p values
# Vehicle Power 8, for example, has a coefficient of 0.0228, which translates to a relativity of 1.023 [e^0.0228]
# This implies there Vehicle Power 8 is only about 2.3% more risky than the refernce, so not much different 

# Now, let's combine some Vehicle Power levels
df_agg = df_agg.copy()

df_agg["VehPower_str"] = df_agg["VehPower"].astype(int).astype(str)

df_agg["VehPower_combined"] = df_agg["VehPower_str"]

df_agg.loc[df_agg["VehPower_combined"].isin(["4", "5", "6", "7"]), "VehPower_combined"] = "4_7"
df_agg.loc[df_agg["VehPower_combined"].isin(["8", "9", "10", "11"]), "VehPower_combined"] = "8_11"
df_agg.loc[df_agg["VehPower_combined"].isin(["12", "13", "14", "15"]), "VehPower_combined"] = "12_15"

# Specify 4 - 7 as reference
df_agg["VehPower_combined"] = pd.Categorical(
    df_agg["VehPower_combined"],
    categories=["4_7", "8_11", "12_15"],  # reference = FIRST
    ordered=True
)

# Check grouping
vehpower_summary = (
    df_agg.groupby("VehPower_combined", observed=True)
    .agg(
        claims=("ClaimNb", "sum"),
        exposure=("Exposure", "sum")
    )
    .reset_index()
)

vehpower_summary["frequency"] = (
    vehpower_summary["claims"] / vehpower_summary["exposure"]
)

print(vehpower_summary)

  VehPower_combined  claims       exposure  frequency
0               4_7   22497  202598.984375   0.111042
1              8_11    4874   43536.777344   0.111951
2             12_15     560    5550.288574   0.100896


In [18]:
# Fit model with grouped Vehicle Power
model_freq_combined = smf.glm(
    formula="""
    ClaimNb ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) + C(VehPower_combined) + C(Area)
    """,
    data=df_agg,
    family=sm.families.NegativeBinomial(alpha=1.0),
    exposure=df_agg["Exposure"]
).fit()

print(model_freq_combined.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:                20109
Model:                            GLM   Df Residuals:                    20086
Model Family:        NegativeBinomial   Df Model:                           22
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -23014.
Date:                Thu, 15 Oct 2026   Deviance:                       11871.
Time:                        00:10:25   Pearson chi2:                 2.78e+04
No. Iterations:                     8   Pseudo R-squ. (CS):            0.09661
Covariance Type:            nonrobust                                         
                                                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------

In [19]:
# Vehicle Power Group 12 - 15 has a coefficient of -0.0201, which translates to a relativity of 0.980 (2% less risky than reference)
# Therefore, it is not significant relative to the reference, and would be combined with 4 - 7

df_agg = df_agg.copy()

# Ensure string form
df_agg["VehPower_str"] = df_agg["VehPower"].astype(int).astype(str)

# Create 2-group structure
df_agg["VehPower_2grp"] = "Base"  # 4–7 and 12–15
df_agg.loc[df_agg["VehPower_str"].isin(["8","9","10","11"]), "VehPower_2grp"] = "High_8_11"

# Set reference = Base
import pandas as pd
df_agg["VehPower_2grp"] = pd.Categorical(
    df_agg["VehPower_2grp"],
    categories=["Base", "High_8_11"],
    ordered=True
)

# Check grouping
vehpower_summary = (
    df_agg.groupby("VehPower_2grp", observed=True)
    .agg(
        claims=("ClaimNb", "sum"),
        exposure=("Exposure", "sum")
    )
    .reset_index()
)

vehpower_summary["frequency"] = (
    vehpower_summary["claims"] / vehpower_summary["exposure"]
)

print(vehpower_summary)

  VehPower_2grp  claims       exposure  frequency
0          Base   23057  208149.265625   0.110771
1     High_8_11    4874   43536.777344   0.111951


In [20]:
# Refit with the combination
model_vp_2grp = smf.glm(
    formula="""
    ClaimNb ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) + C(VehPower_2grp) + C(Area)
    """,
    data=df_agg,
    family=sm.families.NegativeBinomial(alpha=1.0),
    exposure=df_agg["Exposure"]
).fit()

print(model_vp_2grp.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:                20109
Model:                            GLM   Df Residuals:                    20087
Model Family:        NegativeBinomial   Df Model:                           21
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -23014.
Date:                Thu, 15 Oct 2026   Deviance:                       11871.
Time:                        00:10:26   Pearson chi2:                 2.77e+04
No. Iterations:                     8   Pseudo R-squ. (CS):            0.09660
Covariance Type:            nonrobust                                         
                                                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------

In [21]:
# Lastly, let's combine Driver Age Band 35 - 45 with the reference (17 - 25)
print(df_agg["DrivAgeBand"].cat.categories)

# Combine
df_agg = df_agg.copy()

# Convert to string for easy mapping
df_agg["DrivAgeBand_str"] = df_agg["DrivAgeBand"].astype(str)

# Get reference band
ref_band = str(df_agg["DrivAgeBand"].cat.categories[0])

# Replace (35,45] with reference
df_agg.loc[
    df_agg["DrivAgeBand_str"] == "(35, 45]",
    "DrivAgeBand_str"
] = ref_band

# Create new combined variable
df_agg["DrivAgeBand_combined"] = df_agg["DrivAgeBand_str"]

# Check
print(
    df_agg[["DrivAgeBand", "DrivAgeBand_combined"]]
    .drop_duplicates()
    .sort_values("DrivAgeBand_combined")
)

IntervalIndex([(17, 25], (25, 35], (35, 45], (45, 55], (55, 65], (65, 120]], dtype='interval[int64, right]')
   DrivAgeBand DrivAgeBand_combined
2     (35, 45]             (17, 25]
35    (17, 25]             (17, 25]
1     (25, 35]             (25, 35]
7     (45, 55]             (45, 55]
4     (55, 65]             (55, 65]
0    (65, 120]            (65, 120]


In [22]:
# Refit with the new combination
model_freq = smf.glm(
    formula="""
    ClaimNb ~ C(DrivAgeBand_combined) + C(VehAgeBand) + C(BonusMalusBand) + C(VehPower_2grp) + C(Area)
    """,
    data=df_agg,
    family=sm.families.NegativeBinomial(alpha=1.0),
    exposure=df_agg["Exposure"]
).fit()

print(model_freq.summary())

# Results look good from a p-value standpoint

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:                20109
Model:                            GLM   Df Residuals:                    20088
Model Family:        NegativeBinomial   Df Model:                           20
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -23014.
Date:                Thu, 15 Oct 2026   Deviance:                       11872.
Time:                        00:10:27   Pearson chi2:                 2.77e+04
No. Iterations:                     8   Pseudo R-squ. (CS):            0.09659
Covariance Type:            nonrobust                                         
                                                              coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------

In [23]:
# Save frequency model to a Pickle file
model_freq.save("model_freq.pickle")

In [24]:
# Frequency Model Conclusions - Based on coefficients

# Driver Age
    # Younger drivers have a higher risk frequency
    # Middle aged drivers are safer
    # Risk increases again when drivers for older drivers 

# Vehicle Age
    # Older vehicles have a lower risk frequency
    # Likely due to:
        # lower usage
        # different driver behavior regarding older vehicles
        # different claims reporting amongst owners of older vehicles

# Bonus Malus
    # This variable showed exhibits a strong monotonic increase
    # This is an excellent predictor
    # Likely core pricing driver

# Area
    # Clear geographic risk gradient based on Area

# Density
    # This variable was not statistically significant
    # It was removed altogether since it contributed nothing to the modell 